In [10]:
import numpy as np
import cv2 as cv2
import glob

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02], [0, 7.188560e+02,
    1.852157e+02], [0, 0, 1]])

In [11]:
def extract_keypoints_sift(img1, img2, K, baseline):
    """
    Use SIFT to detect keypoints and compute descriptors in both images,
    and find matches between them using knnMatch with k=2
    """
    # sift = ...


    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1,des2,k=2)

    """
    Get the best keypoints by applying the Lowes ratio test
    """
    match_points1 = []
    match_points2 = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:  # Threshold for Lowe's ratio
            match_points1.append(kp1[m.queryIdx].pt)
            match_points2.append(kp2[m.trainIdx].pt)
    
    # Transform best keypoints to nparray
    p1 = np.array(match_points1).astype('float32')
    p2 = np.array(match_points2).astype('float32')

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    # project the feature points to 3D with triangulation
    # projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates (P->Nx4  [N,4] is the normalizer/scale)
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

In [12]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    params = dict(winSize=(21, 21),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    
    #next_points, status, _ = ...

    next_points, status,_ = cv2.calcOpticalFlowPyrLK(prev_img, next_img, prev_points, None, **params)
    idx = np.array(status==1).ravel()
    prev_points = prev_points[idx]
    next_points = next_points[idx]
    world_points = world_points[idx]

    return world_points, prev_points, next_points


In [13]:
def playImageSequence(l_img_paths, r_img_paths, K):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_sift' above
    """
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K, baseline)
    # reference
    reference_img = left_img

    # Groundtruth for plot
    traj = np.zeros((600, 600, 3), dtype=np.uint8)

    for i in range(1, len(l_img_paths)):
        print('image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """
        landmark_3D, reference_2D, current_2D = featureTracking(reference_img, curImage, 
        reference_2D, landmark_3D)

        ##### ####################################### #######
        ##### Calculate relative pose using PNPRansac #######
        ##### ###############################################
        """
        Implement step 2.3)
        """
        _, rvec, tvec,_ = cv2.solvePnPRansac(landmark_3D,current_2D, K, None)
        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame

        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        # re-obtain the 3D points. 
        # hint: use 'extract_keypoints_sift' again
        """
        Implement step 2.4)
        """
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K, baseline)
        #Project the points from camera to world coordinates
        reference_2D = reference_2D_new.astype('float32')
        landmark_3D = inv_transform.dot(np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0])))))
        landmark_3D = landmark_3D.T

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        # draw images
        draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)

        print(tvec[0],tvec[1],tvec[2], rvec[0], rvec[1], rvec[2])
        
        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]),
                                                                           float(tvec[2]))
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj
        vis[:h2, w1:w1 + w2, :3] = np.dstack((np.dstack((curImage,curImage)),curImage))

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break


    cv2.waitKey(0)
    cv2.destroyAllWindows()
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################

In [14]:
# Load image paths
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))

K = getK()

playImageSequence(left_img_paths, right_img_paths, K)

image:  1


C:\Users\anto\AppData\Local\Temp\ipykernel_8784\2930595196.py:74: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  draw_x, draw_y = int(tvec[0]) + 300, 600-(int(tvec[2]) + 100)
C:\Users\anto\AppData\Local\Temp\ipykernel_8784\2930595196.py:78: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(float(tvec[0]), float(tvec[1]),
C:\Users\anto\AppData\Local\Temp\ipykernel_8784\2930595196.py:79: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation.

[-0.01390417] [-0.01556231] [0.626805] [0.00309445] [0.00038945] [0.00184007]
image:  2
[-0.02541636] [-0.0272876] [1.22702149] [0.00588038] [0.00261428] [-1.38014829e-05]
image:  3
[-0.02581965] [-0.04236647] [1.79597211] [0.0055882] [0.00745844] [-0.00591932]
image:  4
[-0.03482258] [-0.05965078] [2.34348958] [0.00379604] [0.01538126] [-0.00943509]
image:  5
[-0.05394732] [-0.07549306] [2.87483641] [0.00027676] [0.02606631] [-0.01152066]
image:  6
[-0.09095049] [-0.08749719] [3.39597082] [-0.00317595] [0.03999151] [-0.01173086]
image:  7
[-0.13754904] [-0.10062936] [3.8980101] [-0.00526073] [0.05869167] [-0.01177135]
image:  8
[-0.19160066] [-0.11288775] [4.40208541] [-0.00388522] [0.08056396] [-0.01501341]
image:  9
[-0.26475514] [-0.12840167] [4.90217425] [-0.00073826] [0.10512208] [-0.01213134]
image:  10
[-0.34710033] [-0.146094] [5.38431917] [0.00261004] [0.13331463] [-0.01155015]
image:  11
[-0.44083593] [-0.15600599] [5.8612969] [0.00304709] [0.16601866] [-0.01300131]
image:  